# Week 9 - Morfologi Citra

Praktikum morfologi matematis: dilasi, erosi, opening, closing, top-hat, bottom-hat,
skeleton, thinning, dan penerapannya sebagai praolah OCR pada citra KTP.

Seluruh angka pada laporan dibaca dari `results.json` yang ditulis sel terakhir.

In [1]:
# Setup: impor, jalur, dan wadah hasil
import os, json, time
from datetime import datetime, timezone, timedelta

import cv2
import numpy as np
from matplotlib import pyplot as plt
import pytesseract
from skimage import filters, morphology as skmorph, measure

pytesseract.pytesseract.tesseract_cmd = r'C:/Program Files/Tesseract-OCR/tesseract.exe'

BASE = os.getcwd()
IMG  = os.path.join(BASE, 'images')
OUT  = os.path.join(BASE, 'out')
TESS = os.path.join(BASE, 'tessdata')
KTP_DIR = os.path.abspath(os.path.join(BASE, '..', 'Week 7'))
os.makedirs(OUT, exist_ok=True)
if os.path.isdir(TESS):
    os.environ['TESSDATA_PREFIX'] = TESS

import platform, skimage
R = {'nama': 'Adam Bahy Maulana', 'nim': '244107020207', 'N': 207,
     'lingkungan': {'python': platform.python_version(), 'opencv': cv2.__version__,
                    'numpy': np.__version__, 'skimage': skimage.__version__,
                    'tesseract': str(pytesseract.get_tesseract_version())},
     'waktu': datetime.now(timezone(timedelta(hours=7))).isoformat(timespec='seconds'),
     'citra': {}, 'dilasi': {}, 'erosi': {}, 'gabungan': {}, 'lanjut': {}, 'ktp': {}}


def simpan(fig, nama):
    if isinstance(fig, str):
        return fig
    fig.savefig(os.path.join(OUT, nama), dpi=300, bbox_inches='tight')
    plt.close(fig)
    return nama


def tampil(daftar, judul, nama, cmap='gray', figsize=(11, 3.4)):
    fig, ax = plt.subplots(1, len(daftar), figsize=figsize)
    for a, (g, t) in zip(np.atleast_1d(ax), daftar):
        a.imshow(g, cmap=cmap); a.set_title(t, fontsize=9)
        a.set_xticks([]); a.set_yticks([])
    fig.suptitle(judul, fontsize=10)
    return simpan(fig, nama)


def modul_beda(fn, F, k, se, ref):
    # Kode modul memakai jendela persegi k x k, jadi SE tak persegi bisa gagal
    # di-broadcast. Kegagalan itu sendiri dicatat sebagai temuan.
    try:
        return beda(fn(F, k, se), ref)
    except Exception as e:
        return f'{type(e).__name__}: {e}'


def putih(img):
    return int((img > 0).sum())


def proporsi(img):
    return round(float((img > 0).mean()), 4)


def beda(a, b, cr=0, cc=None):
    # Jumlah piksel berbeda; cr/cc = lebar tepi yang dikecualikan per sumbu.
    cc = cr if cc is None else cc
    if cr or cc:
        a = a[cr:a.shape[0]-cr, cc:a.shape[1]-cc]
        b = b[cr:b.shape[0]-cr, cc:b.shape[1]-cc]
    return int((a != b).sum())

In [2]:
# Citra kerja. Modul menunjuk 'plat nomer.jpg' yang tidak ada pada folder bahan,
# jadi dipakai 'sudoku-original.jpg' (citra teks bergaris tipis, karakter setara).
plat_bgr = cv2.imread(os.path.join(IMG, 'sudoku-original.jpg'))
plat = cv2.cvtColor(plat_bgr, cv2.COLOR_BGR2GRAY)
# Modul memakai THRESH_BINARY sehingga yang bernilai 1 adalah latar; di sini dibalik
# agar objek (goresan tinta) yang bernilai 1. Pencahayaan halaman tidak rata
# (bayangan di tepi), sehingga citra dibagi dulu dengan latar hasil blur sebelum
# Otsu. Tanpa langkah ini bayangan ikut menjadi objek dan proporsi objek melonjak.
latar = cv2.GaussianBlur(plat, (0, 0), 25)
rata = cv2.divide(plat, latar, scale=255)
ambang, biner = cv2.threshold(rata, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

R['citra']['plat'] = {'berkas': 'sudoku-original.jpg', 'shape': list(plat.shape),
                      'ambang_otsu': int(ambang), 'proporsi_objek': proporsi(biner),
                      'proporsi_objek_tanpa_perataan': proporsi(
                          cv2.threshold(plat, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1])}

fig = tampil([(plat, 'Citra awal (grayscale)'), (rata, 'Pencahayaan diratakan'),
              (biner, 'Biner, objek = putih')],
             'Citra kerja dan hasil ambang', '00_citra_kerja.png')
plt.show()
print('shape', plat.shape, R['citra']['plat'])

shape (423, 419) {'berkas': 'sudoku-original.jpg', 'shape': [423, 419], 'ambang_otsu': 181, 'proporsi_objek': 0.1116, 'proporsi_objek_tanpa_perataan': 0.4053}


In [3]:
# Dua implementasi: mengikuti kode modul apa adanya (jendela persegi), dan versi
# yang jendelanya mengikuti bentuk SE sehingga sama dengan hasil library.
def dilasi_modul(F, w, se=None):
    p, q = F.shape
    SED = np.ones((w, w), dtype=np.uint8) if se is None else se
    c = (w - 1) // 2
    imgD = np.zeros((p, q), dtype=np.uint8)
    for i in range(c, p - c):
        for j in range(c, q - c):
            imgD[i, j] = np.max(F[i-c:i+c+1, j-c:j+c+1] * SED)
    return imgD


def erosi_modul(F, k, se=None):
    m, n = F.shape
    SE = np.ones((k, k), dtype=np.uint8) if se is None else se
    c = (k - 1) // 2
    imgE = np.zeros((m, n), dtype=np.uint8)
    for i in range(c, m - c):
        for j in range(c, n - c):
            imgE[i, j] = np.min(F[i-c:i+c+1, j-c:j+c+1] * SE)
    return imgE


def dilasi_se(F, se):
    p, q = F.shape
    h, w = se.shape[0] // 2, se.shape[1] // 2
    out = np.zeros((p, q), np.uint8)
    for i in range(h, p - h):
        for j in range(w, q - w):
            out[i, j] = np.max(F[i-h:i+h+1, j-w:j+w+1] * se)
    return out


def erosi_se(F, se):
    m, n = F.shape
    h, w = se.shape[0] // 2, se.shape[1] // 2
    out = np.zeros((m, n), np.uint8)
    for i in range(h, m - h):
        for j in range(w, n - w):
            out[i, j] = np.min(F[i-h:i+h+1, j-w:j+w+1] * se)
    return out


print('fungsi siap')

fungsi siap


In [4]:
# Langkah 3 - dilasi tanpa library, SE square 5x5
t0 = time.perf_counter()
dil_man5 = dilasi_modul(biner, 5)
t_dil_man = time.perf_counter() - t0

R['dilasi']['langkah3'] = {'se': 'square 5x5 (np.ones)', 'waktu_manual_ms': round(t_dil_man*1000, 1),
                           'objek_awal': putih(biner), 'objek_hasil': putih(dil_man5),
                           'pertambahan_persen': round((putih(dil_man5)/putih(biner)-1)*100, 2)}

tampil([(plat, 'Citra awal'), (biner, 'Biner'), (dil_man5, 'Dilasi manual 5x5')],
       'Langkah 3 - dilasi manual, SE square 5x5', '01_dilasi_manual.png')
plt.show()
print(R['dilasi']['langkah3'])

{'se': 'square 5x5 (np.ones)', 'waktu_manual_ms': 1317.2, 'objek_awal': 19784, 'objek_hasil': 54167, 'pertambahan_persen': 173.79}


In [5]:
# Langkah 4 - dilasi dengan library OpenCV, SE square 3x3
t0 = time.perf_counter()
dil_lib3 = cv2.dilate(biner, np.ones((3, 3), np.uint8), iterations=1)
t_dil_lib = time.perf_counter() - t0
dil_man3 = dilasi_modul(biner, 3)

R['dilasi']['langkah4'] = {
    'se': 'square 3x3', 'waktu_library_ms': round(t_dil_lib*1000, 2),
    'percepatan_kali': round(t_dil_man / max(t_dil_lib, 1e-9), 1),
    'beda_interior_piksel': beda(dil_lib3, dil_man3, 1),
    'beda_total_piksel': beda(dil_lib3, dil_man3),
    'objek_hasil': putih(dil_lib3)}

tampil([(biner, 'Citra biner'), (dil_man3, 'Dilasi manual 3x3'), (dil_lib3, 'Dilasi cv2 3x3')],
       'Langkah 4 - dilasi manual vs library, SE square 3x3', '02_dilasi_library.png')
plt.show()
print(R['dilasi']['langkah4'])

{'se': 'square 3x3', 'waktu_library_ms': 0.28, 'percepatan_kali': 4779.5, 'beda_interior_piksel': 0, 'beda_total_piksel': 180, 'objek_hasil': 38047}


In [6]:
# Langkah 5 - erosi tanpa library, SE square 5x5
t0 = time.perf_counter()
ero_man5 = erosi_modul(biner, 5)
t_ero_man = time.perf_counter() - t0

R['erosi']['langkah5'] = {'se': 'square 5x5', 'waktu_manual_ms': round(t_ero_man*1000, 1),
                          'objek_awal': putih(biner), 'objek_hasil': putih(ero_man5),
                          'penyusutan_persen': round((1-putih(ero_man5)/putih(biner))*100, 2)}

tampil([(plat, 'Citra awal'), (biner, 'Biner'), (ero_man5, 'Erosi manual 5x5')],
       'Langkah 5 - erosi manual, SE square 5x5', '03_erosi_manual.png')
plt.show()
print(R['erosi']['langkah5'])

{'se': 'square 5x5', 'waktu_manual_ms': 2107.9, 'objek_awal': 19784, 'objek_hasil': 1246, 'penyusutan_persen': 93.7}


In [7]:
# Langkah 6 - erosi dengan library OpenCV, SE square 5x5
t0 = time.perf_counter()
ero_lib5 = cv2.erode(biner, np.ones((5, 5), np.uint8), iterations=1)
t_ero_lib = time.perf_counter() - t0

R['erosi']['langkah6'] = {
    'se': 'square 5x5', 'waktu_library_ms': round(t_ero_lib*1000, 2),
    'percepatan_kali': round(t_ero_man / max(t_ero_lib, 1e-9), 1),
    'beda_interior_piksel': beda(ero_lib5, ero_man5, 2),
    'beda_total_piksel': beda(ero_lib5, ero_man5),
    'objek_hasil': putih(ero_lib5)}

tampil([(biner, 'Citra biner'), (ero_man5, 'Erosi manual 5x5'), (ero_lib5, 'Erosi cv2 5x5')],
       'Langkah 6 - erosi manual vs library, SE square 5x5', '04_erosi_library.png')
plt.show()
print(R['erosi']['langkah6'])

{'se': 'square 5x5', 'waktu_library_ms': 0.27, 'percepatan_kali': 7679.0, 'beda_interior_piksel': 0, 'beda_total_piksel': 81, 'objek_hasil': 1327}


In [8]:
# Langkah 7 - opening, tanpa library dan dengan library, SE square 3x3
t0 = time.perf_counter()
open_man = dilasi_modul(erosi_modul(biner, 3), 3)
t_open_man = time.perf_counter() - t0
t0 = time.perf_counter()
open_lib = cv2.morphologyEx(biner, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
t_open_lib = time.perf_counter() - t0
open_dua = cv2.morphologyEx(open_lib, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

R['gabungan']['langkah7'] = {
    'se': 'square 3x3', 'objek_awal': putih(biner), 'objek_manual': putih(open_man),
    'objek_library': putih(open_lib),
    'selisih_persen_vs_awal': round((putih(open_lib)/putih(biner)-1)*100, 2),
    'beda_interior_manual_vs_library': beda(open_lib, open_man, 2),
    'beda_tepi_manual_vs_library': beda(open_lib, open_man),
    'idempoten_beda_piksel': beda(open_dua, open_lib),
    'waktu_manual_ms': round(t_open_man*1000, 1), 'waktu_library_ms': round(t_open_lib*1000, 2)}

tampil([(biner, 'Citra biner'), (open_man, 'Opening manual'), (open_lib, 'Opening cv2')],
       'Langkah 7 - opening manual vs library, SE square 3x3', '05_opening.png')
plt.show()
print(R['gabungan']['langkah7'])

{'se': 'square 3x3', 'objek_awal': 19784, 'objek_manual': 14235, 'objek_library': 14304, 'selisih_persen_vs_awal': -27.7, 'beda_interior_manual_vs_library': 0, 'beda_tepi_manual_vs_library': 69, 'idempoten_beda_piksel': 0, 'waktu_manual_ms': 3043.7, 'waktu_library_ms': 0.24}


In [9]:
# Langkah 8 - closing, tanpa library dan dengan library, SE square 3x3
t0 = time.perf_counter()
clos_man = erosi_modul(dilasi_modul(biner, 3), 3)
t_clos_man = time.perf_counter() - t0
t0 = time.perf_counter()
clos_lib = cv2.morphologyEx(biner, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
t_clos_lib = time.perf_counter() - t0
clos_dua = cv2.morphologyEx(clos_lib, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))

# teorema: opening(A) subset A subset closing(A)
subset_buka = int((open_lib & ~biner).sum()) == 0
subset_tutup = int((biner & ~clos_lib).sum()) == 0

R['gabungan']['langkah8'] = {
    'se': 'square 3x3', 'objek_awal': putih(biner), 'objek_manual': putih(clos_man),
    'objek_library': putih(clos_lib),
    'selisih_persen_vs_awal': round((putih(clos_lib)/putih(biner)-1)*100, 2),
    'beda_interior_manual_vs_library': beda(clos_lib, clos_man, 2),
    'beda_tepi_manual_vs_library': beda(clos_lib, clos_man),
    'idempoten_beda_piksel': beda(clos_dua, clos_lib),
    'opening_subset_citra': bool(subset_buka), 'citra_subset_closing': bool(subset_tutup),
    'waktu_manual_ms': round(t_clos_man*1000, 1), 'waktu_library_ms': round(t_clos_lib*1000, 2)}

tampil([(biner, 'Citra biner'), (clos_man, 'Closing manual'), (clos_lib, 'Closing cv2')],
       'Langkah 8 - closing manual vs library, SE square 3x3', '06_closing.png')
plt.show()
print(R['gabungan']['langkah8'])

{'se': 'square 3x3', 'objek_awal': 19784, 'objek_manual': 21274, 'objek_library': 21511, 'selisih_persen_vs_awal': 8.73, 'beda_interior_manual_vs_library': 0, 'beda_tepi_manual_vs_library': 237, 'idempoten_beda_piksel': 0, 'opening_subset_citra': True, 'citra_subset_closing': True, 'waktu_manual_ms': 2463.9, 'waktu_library_ms': 0.4}


In [10]:
# Langkah 9 - dilasi SE cross 3x3 dan 5x5
cross3 = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], np.uint8)
cross5 = np.array([[0, 0, 1, 0, 0], [0, 0, 1, 0, 0], [1, 1, 1, 1, 1],
                   [0, 0, 1, 0, 0], [0, 0, 1, 0, 0]], np.uint8)
dil_cross3 = dilasi_se(biner, cross3)
dil_cross5 = dilasi_se(biner, cross5)

R['dilasi']['langkah9'] = {
    'cross3': {'objek': putih(dil_cross3),
               'pertambahan_persen': round((putih(dil_cross3)/putih(biner)-1)*100, 2),
               'beda_interior_vs_cv2': beda(cv2.dilate(biner, cross3), dil_cross3, 1),
               'beda_kode_modul_piksel': modul_beda(dilasi_modul, biner, 3, cross3, dil_cross3)},
    'cross5': {'objek': putih(dil_cross5),
               'pertambahan_persen': round((putih(dil_cross5)/putih(biner)-1)*100, 2),
               'beda_interior_vs_cv2': beda(cv2.dilate(biner, cross5), dil_cross5, 2),
               'beda_kode_modul_piksel': modul_beda(dilasi_modul, biner, 5, cross5, dil_cross5)}}

tampil([(biner, 'Biner'), (dil_cross3, 'Cross 3x3'), (dil_cross5, 'Cross 5x5')],
       'Langkah 9 - dilasi SE cross 3x3 dan 5x5', '07_dilasi_cross.png')
plt.show()
print(R['dilasi']['langkah9'])

{'cross3': {'objek': 34453, 'pertambahan_persen': 74.15, 'beda_interior_vs_cv2': 0, 'beda_kode_modul_piksel': 0}, 'cross5': {'objek': 47140, 'pertambahan_persen': 138.27, 'beda_interior_vs_cv2': 0, 'beda_kode_modul_piksel': 0}}


In [11]:
# Langkah 10 - dilasi SE circular (ellipse) 3x3 dan 5x5
circ3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
circ5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
dil_circ3 = dilasi_se(biner, circ3)
dil_circ5 = dilasi_se(biner, circ5)

R['dilasi']['langkah10'] = {
    'circ3': {'kernel': circ3.tolist(), 'objek': putih(dil_circ3),
              'pertambahan_persen': round((putih(dil_circ3)/putih(biner)-1)*100, 2),
              'beda_interior_vs_cv2': beda(cv2.dilate(biner, circ3), dil_circ3, 1),
              'beda_kode_modul_piksel': modul_beda(dilasi_modul, biner, 3, circ3, dil_circ3)},
    'circ5': {'kernel': circ5.tolist(), 'objek': putih(dil_circ5),
              'pertambahan_persen': round((putih(dil_circ5)/putih(biner)-1)*100, 2),
              'beda_interior_vs_cv2': beda(cv2.dilate(biner, circ5), dil_circ5, 2),
              'beda_kode_modul_piksel': modul_beda(dilasi_modul, biner, 5, circ5, dil_circ5)}}

tampil([(biner, 'Biner'), (dil_circ3, 'Circular 3x3'), (dil_circ5, 'Circular 5x5')],
       'Langkah 10 - dilasi SE circular 3x3 dan 5x5', '08_dilasi_circular.png')
plt.show()
print(R['dilasi']['langkah10'])

{'circ3': {'kernel': [[0, 1, 0], [1, 1, 1], [0, 1, 0]], 'objek': 34453, 'pertambahan_persen': 74.15, 'beda_interior_vs_cv2': 0, 'beda_kode_modul_piksel': 0}, 'circ5': {'kernel': [[0, 0, 1, 0, 0], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [0, 0, 1, 0, 0]], 'objek': 50500, 'pertambahan_persen': 155.26, 'beda_interior_vs_cv2': 0, 'beda_kode_modul_piksel': 0}}


In [12]:
# Perbandingan bentuk SE pada dilasi ukuran sama: square, cross, circular
tabel_se = []
for w in (3, 5):
    for nama, se in (('square', np.ones((w, w), np.uint8)),
                     ('cross', cross3 if w == 3 else cross5),
                     ('circular', cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (w, w)))):
        hasil = dilasi_se(biner, se)
        tabel_se.append({'ukuran': w, 'bentuk': nama, 'jumlah_piksel_kernel': int(se.sum()),
                         'objek_hasil': putih(hasil),
                         'pertambahan_persen': round((putih(hasil)/putih(biner)-1)*100, 2)})
R['dilasi']['perbandingan_se'] = tabel_se

fig, ax = plt.subplots(2, 3, figsize=(9, 5.6))
pasangan = [(3, 'square', np.ones((3, 3), np.uint8)), (3, 'cross', cross3),
            (3, 'circular', circ3), (5, 'square', np.ones((5, 5), np.uint8)),
            (5, 'cross', cross5), (5, 'circular', circ5)]
for a, (w, nama, se) in zip(ax.ravel(), pasangan):
    a.imshow(dilasi_se(biner, se), cmap='gray')
    a.set_title(f'{nama} {w}x{w}', fontsize=9); a.set_xticks([]); a.set_yticks([])
fig.suptitle('Pengaruh bentuk structuring element pada dilasi', fontsize=10)
simpan(fig, '09_perbandingan_se.png')
plt.show()
print(tabel_se)

[{'ukuran': 3, 'bentuk': 'square', 'jumlah_piksel_kernel': 9, 'objek_hasil': 37867, 'pertambahan_persen': 91.4}, {'ukuran': 3, 'bentuk': 'cross', 'jumlah_piksel_kernel': 5, 'objek_hasil': 34453, 'pertambahan_persen': 74.15}, {'ukuran': 3, 'bentuk': 'circular', 'jumlah_piksel_kernel': 5, 'objek_hasil': 34453, 'pertambahan_persen': 74.15}, {'ukuran': 5, 'bentuk': 'square', 'jumlah_piksel_kernel': 25, 'objek_hasil': 54167, 'pertambahan_persen': 173.79}, {'ukuran': 5, 'bentuk': 'cross', 'jumlah_piksel_kernel': 9, 'objek_hasil': 47140, 'pertambahan_persen': 138.27}, {'ukuran': 5, 'bentuk': 'circular', 'jumlah_piksel_kernel': 17, 'objek_hasil': 50500, 'pertambahan_persen': 155.26}]


In [13]:
# Langkah 11 - erosi SE rectangle 3x5 dan 5x7
rect35 = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 3))   # (kolom, baris)
rect57 = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 5))
ero_r35 = erosi_se(biner, rect35)
ero_r57 = erosi_se(biner, rect57)

R['erosi']['langkah11'] = {
    'rect3x5': {'objek': putih(ero_r35), 'penyusutan_persen': round((1-putih(ero_r35)/putih(biner))*100, 2),
                'beda_interior_vs_cv2': beda(cv2.erode(biner, rect35), ero_r35, 1, 2),
                'beda_kode_modul_piksel': modul_beda(erosi_modul, biner, 5, rect35, ero_r35)},
    'rect5x7': {'objek': putih(ero_r57), 'penyusutan_persen': round((1-putih(ero_r57)/putih(biner))*100, 2),
                'beda_interior_vs_cv2': beda(cv2.erode(biner, rect57), ero_r57, 2, 3),
                'beda_kode_modul_piksel': modul_beda(erosi_modul, biner, 7, rect57, ero_r57)}}

tampil([(biner, 'Biner'), (ero_r35, 'Rect 3x5'), (ero_r57, 'Rect 5x7')],
       'Langkah 11 - erosi SE rectangle 3x5 dan 5x7', '10_erosi_rect.png')
plt.show()
print(R['erosi']['langkah11'])

{'rect3x5': {'objek': 3107, 'penyusutan_persen': 84.3, 'beda_interior_vs_cv2': 0, 'beda_kode_modul_piksel': 'ValueError: operands could not be broadcast together with shapes (5,5) (3,5) '}, 'rect5x7': {'objek': 1133, 'penyusutan_persen': 94.27, 'beda_interior_vs_cv2': 0, 'beda_kode_modul_piksel': 'ValueError: operands could not be broadcast together with shapes (7,7) (5,7) '}}


In [14]:
# Langkah 12 - erosi SE line vertikal 3 dan 5 pada j.png
j_gray = cv2.imread(os.path.join(IMG, 'j.png'), 0)
# j.png sudah biner dengan glif putih (15,3% piksel terang), jadi objek = putih.
# Erosi dihitung pada resolusi asli agar SE 3 dan 5 bermakna; hasilnya diperbesar
# 4x tanpa interpolasi hanya untuk keperluan tampilan.
_, j_bin = cv2.threshold(j_gray, 127, 255, cv2.THRESH_BINARY)

line3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (1, 3))
line5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (1, 5))
j_l3 = erosi_se(j_bin, line3)
j_l5 = erosi_se(j_bin, line5)


def besar(img, f=4):
    return cv2.resize(img, None, fx=f, fy=f, interpolation=cv2.INTER_NEAREST)

R['erosi']['langkah12'] = {
    'berkas': 'j.png', 'shape': list(j_gray.shape), 'objek_awal': putih(j_bin),
    'line3': {'objek': putih(j_l3), 'sisa_persen': round(putih(j_l3)/putih(j_bin)*100, 2),
              'beda_interior_vs_cv2': beda(cv2.erode(j_bin, line3), j_l3, 1, 0),
              'beda_kode_modul_piksel': modul_beda(erosi_modul, j_bin, 3, line3, j_l3)},
    'line5': {'objek': putih(j_l5), 'sisa_persen': round(putih(j_l5)/putih(j_bin)*100, 2),
              'beda_interior_vs_cv2': beda(cv2.erode(j_bin, line5), j_l5, 2, 0),
              'beda_kode_modul_piksel': modul_beda(erosi_modul, j_bin, 5, line5, j_l5)}}

tampil([(besar(j_bin), 'j.png biner'), (besar(j_l3), 'Line vertikal 3'), (besar(j_l5), 'Line vertikal 5')],
       'Langkah 12 - erosi SE line vertikal 3 dan 5', '11_erosi_line.png', figsize=(9, 3.4))
plt.show()
print(R['erosi']['langkah12'])

{'berkas': 'j.png', 'shape': [150, 112], 'objek_awal': 2576, 'line3': {'objek': 2175, 'sisa_persen': 84.43, 'beda_interior_vs_cv2': 0, 'beda_kode_modul_piksel': 384}, 'line5': {'objek': 1775, 'sisa_persen': 68.91, 'beda_interior_vs_cv2': 0, 'beda_kode_modul_piksel': 760}}


In [15]:
# Langkah 13 - top-hat pada citra teks
kern_th = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
tophat = cv2.morphologyEx(plat, cv2.MORPH_TOPHAT, kern_th)
# verifikasi definisi: top-hat = citra - opening(citra)
tophat_manual = cv2.subtract(plat, cv2.morphologyEx(plat, cv2.MORPH_OPEN, kern_th))

R['lanjut']['langkah13'] = {
    'kernel': 'rect 3x3', 'rata_awal': round(float(plat.mean()), 2), 'rata_tophat': round(float(tophat.mean()), 2),
    'piksel_terang_baru': int((tophat > 30).sum()),
    'beda_definisi_piksel': int((tophat.astype(int) - tophat_manual.astype(int) != 0).sum())}

tampil([(plat, 'Citra awal'), (tophat, 'Top-hat (white)')],
       'Langkah 13 - top-hat, SE rect 3x3', '12_tophat.png', figsize=(7, 3.4))
plt.show()
print(R['lanjut']['langkah13'])

{'kernel': 'rect 3x3', 'rata_awal': 101.85, 'rata_tophat': 1.73, 'piksel_terang_baru': 880, 'beda_definisi_piksel': 0}


In [16]:
# Langkah 14 - bottom-hat (black-hat) pada kitten01.jpg
kit = cv2.imread(os.path.join(IMG, 'kitten01.jpg'), 0)
kern_bh = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 5))
blackhat = cv2.morphologyEx(kit, cv2.MORPH_BLACKHAT, kern_bh)
blackhat_manual = cv2.subtract(cv2.morphologyEx(kit, cv2.MORPH_CLOSE, kern_bh), kit)

R['lanjut']['langkah14'] = {
    'kernel': 'rect 13x5', 'shape': list(kit.shape), 'rata_awal': round(float(kit.mean()), 2),
    'rata_blackhat': round(float(blackhat.mean()), 2), 'piksel_gelap_baru': int((blackhat > 30).sum()),
    'beda_definisi_piksel': int((blackhat.astype(int) - blackhat_manual.astype(int) != 0).sum())}

tampil([(kit, 'Citra awal'), (blackhat, 'Bottom-hat (black)')],
       'Langkah 14 - bottom-hat, SE rect 13x5', '13_blackhat.png', figsize=(7, 3.4))
plt.show()
print(R['lanjut']['langkah14'])

{'kernel': 'rect 13x5', 'shape': [627, 670], 'rata_awal': 98.07, 'rata_blackhat': 8.73, 'piksel_gelap_baru': 28925, 'beda_definisi_piksel': 0}


In [17]:
# Langkah 15 dan 16 - skeleton dan skeleton inverse pada lily.jpg
lily = cv2.imread(os.path.join(IMG, 'lily.jpg'), 0)
ambang_tri = filters.threshold_triangle(lily)
bin_lily = (lily > ambang_tri).astype(np.uint8)
skel = skmorph.skeletonize(bin_lily.astype(bool))

bin_inv = lily < ambang_tri
skel_inv = skmorph.skeletonize(bin_inv)

R['lanjut']['langkah15'] = {
    'berkas': 'lily.jpg', 'shape': list(lily.shape), 'ambang_triangle': round(float(ambang_tri), 2),
    'objek_biner': int(bin_lily.sum()), 'piksel_skeleton': int(skel.sum()),
    'kompresi_persen': round((1 - skel.sum()/max(bin_lily.sum(), 1))*100, 2),
    'komponen_skeleton': int(measure.label(skel, connectivity=2).max()),
    'komponen_biner': int(measure.label(bin_lily.astype(bool), connectivity=2).max())}
R['lanjut']['langkah16'] = {
    'objek_biner_inverse': int(bin_inv.sum()), 'piksel_skeleton_inverse': int(skel_inv.sum()),
    'kompresi_persen': round((1 - skel_inv.sum()/max(bin_inv.sum(), 1))*100, 2)}

fig, ax = plt.subplots(1, 4, figsize=(13, 3.4))
for a, g, t in zip(ax, [lily, skel, bin_inv, skel_inv],
                   ['Citra awal', 'Skeleton (objek terang)', 'Biner inverse', 'Skeleton inverse']):
    a.imshow(g, cmap='gray_r' if 'inverse' in t else 'gray')
    a.set_title(t, fontsize=9); a.set_xticks([]); a.set_yticks([])
fig.suptitle('Langkah 15 dan 16 - skeleton dan skeleton inverse', fontsize=10)
simpan(fig, '14_skeleton.png')
plt.show()
print(R['lanjut']['langkah15'], R['lanjut']['langkah16'])

{'berkas': 'lily.jpg', 'shape': [599, 623], 'ambang_triangle': 253.0, 'objek_biner': 212716, 'piksel_skeleton': 15477, 'kompresi_persen': np.float64(92.72), 'komponen_skeleton': 164, 'komponen_biner': 164} {'objek_biner_inverse': 158865, 'piksel_skeleton_inverse': 18916, 'kompresi_persen': np.float64(88.09)}


In [18]:
# Langkah 17 - thickening: skeletonize vs thin
img_f = lily.astype(np.float32) / 255.0
bin_f = img_f < 0.5
out_skel = skmorph.skeletonize(bin_f)
out_thin = skmorph.thin(bin_f, max_num_iter=None)

R['lanjut']['langkah17'] = {
    'objek_biner': int(bin_f.sum()), 'piksel_skeletonize': int(out_skel.sum()),
    'piksel_thin': int(out_thin.sum()),
    'selisih_piksel': int(abs(int(out_skel.sum()) - int(out_thin.sum()))),
    'identik': bool(np.array_equal(out_skel, out_thin)),
    'komponen_thin': int(measure.label(out_thin, connectivity=2).max())}

fig, ax = plt.subplots(1, 3, figsize=(10, 3.4))
for a, g, t in zip(ax, [bin_f, out_skel, out_thin], ['Biner (gelap = objek)', 'skeletonize', 'thin']):
    a.imshow(g, cmap='gray'); a.set_title(t, fontsize=9); a.set_xticks([]); a.set_yticks([])
fig.suptitle('Langkah 17 - thickening: skeletonize vs thin', fontsize=10)
simpan(fig, '15_thin.png')
plt.show()
print(R['lanjut']['langkah17'])

{'objek_biner': 24736, 'piksel_skeletonize': 8998, 'piksel_thin': 8929, 'selisih_piksel': 69, 'identik': False, 'komponen_thin': 918}


## Tugas Praktikum - morfologi pada citra KTP

Empat citra KTP Modul 7 diproses dengan lima varian praolah lalu dibaca OCR. Ukuran
keberhasilan: rerata keyakinan kata (`conf`), jumlah kata terbaca, dan proporsi kata
acuan yang ditemukan pada hasil OCR (acuan = kolom `acuan` pada `tabel_ocr.json`
Modul 7). Proporsi acuan dipakai sebagai ukuran utama karena tahan terhadap kelebihan
teks yang tidak diminta, berbeda dengan CER yang dapat melebihi 1,0 bila keluaran jauh
lebih panjang daripada acuan.

In [19]:
# Tugas - pipeline morfologi pada KTP + OCR sebelum/sesudah
acuan = json.load(open(os.path.join(KTP_DIR, 'tabel_ocr.json'), encoding='utf-8'))
BERKAS = ['ktp_adam.jpg', 'ktp_fulan.jpg', 'ktp_fulani.jpg', 'ktp_jack.jpg']


def ocr(img, psm=6, lang='ind'):
    d = pytesseract.image_to_data(img, lang=lang, config=f'--psm {psm}',
                                  output_type=pytesseract.Output.DICT)
    kata = [(t.strip(), int(c)) for t, c in zip(d['text'], d['conf']) if t.strip() and int(c) > 0]
    if not kata:
        return '', 0.0, 0
    return ' '.join(t for t, _ in kata), round(float(np.mean([c for _, c in kata])), 2), len(kata)


def temuan_kata(teks, daftar_acuan):
    teks = ' ' + ' '.join(teks.upper().split()) + ' '
    hit = sum(1 for a in daftar_acuan if f' {a.upper()} ' in teks)
    return hit, len(daftar_acuan)


def bersihkan(up):
    # up: citra kanal terpilih, sudah diperbesar. Objek (tinta) dijadikan putih.
    _, otsu = cv2.threshold(up, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    fg = cv2.bitwise_not(otsu)
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))   # tutup lubang goresan
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))    # buang bintik noise
    return cv2.bitwise_not(fg)


def varian(gray, bgr):
    up = cv2.resize(gray, None, fx=1.6, fy=1.6, interpolation=cv2.INTER_CUBIC)
    kanal = cv2.resize(bgr[:, :, 0], None, fx=1.6, fy=1.6, interpolation=cv2.INTER_CUBIC)  # biru
    _, otsu = cv2.threshold(up, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    bersih = bersihkan(up)
    bg = cv2.morphologyEx(up, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25)))
    flat = cv2.add(cv2.subtract(up, bg), 128)                               # top-hat: ratakan latar
    _, otsu_flat = cv2.threshold(flat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    fgf = cv2.bitwise_not(otsu_flat)
    fgf = cv2.morphologyEx(fgf, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    bersih_flat = cv2.bitwise_not(fgf)
    adapt = cv2.adaptiveThreshold(up, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY, 101, 35)               # jendela besar: tahan bayangan
    fga = cv2.bitwise_not(adapt)
    fga = cv2.morphologyEx(fga, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
    bersih_adapt = cv2.bitwise_not(fga)
    return {'A_grayscale': up, 'B_otsu': otsu, 'C_otsu_morfologi': bersih,
            'D_tophat_otsu': otsu_flat, 'E_tophat_otsu_morfologi': bersih_flat,
            'F_adaptive_morfologi': bersih_adapt, 'G_kanal_biru_morfologi': bersihkan(kanal)}


def ukur(img, daftar):
    teks, conf, jml = ocr(img)
    hit, total = temuan_kata(teks, daftar)
    return {'conf': conf, 'kata': jml, 'kata_acuan_ditemukan': hit, 'kata_acuan_total': total,
            'proporsi_acuan': round(hit / total, 4), 'cuplikan': teks[:150]}


hasil_ktp, ablr = {}, {}
for berkas in BERKAS:
    bgr = cv2.imread(os.path.join(KTP_DIR, berkas))
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    daftar = [r['acuan'] for r in acuan[berkas]['rows']]
    baris = {}
    for nama, img in varian(gray, bgr).items():
        t0 = time.perf_counter()
        baris[nama] = ukur(img, daftar)
        baris[nama]['waktu_ms'] = round((time.perf_counter() - t0) * 1000, 1)
    # ablasi kanal warna: pipeline morfologi yang sama, kanal B/G/R
    ablr[berkas] = {}
    for i, ch in enumerate('BGR'):
        up = cv2.resize(bgr[:, :, i], None, fx=1.6, fy=1.6, interpolation=cv2.INTER_CUBIC)
        ablr[berkas][ch] = ukur(bersihkan(up), daftar)
    awal = baris['A_grayscale']
    terbaik = max((k for k in baris if k != 'A_grayscale'),
                  key=lambda k: (baris[k]['proporsi_acuan'], baris[k]['conf']))
    hasil_ktp[berkas] = {'shape': list(gray.shape), 'varian': baris, 'terbaik': terbaik,
                         'delta_conf': round(baris[terbaik]['conf'] - awal['conf'], 2),
                         'delta_acuan': round(baris[terbaik]['proporsi_acuan'] - awal['proporsi_acuan'], 4)}
R['ktp'], R['ktp_kanal'] = hasil_ktp, ablr
print(json.dumps({k: {'terbaik': v['terbaik'], 'd_conf': v['delta_conf'], 'd_acuan': v['delta_acuan']}
                  for k, v in hasil_ktp.items()}, indent=1))

{
 "ktp_adam.jpg": {
  "terbaik": "G_kanal_biru_morfologi",
  "d_conf": 32.34,
  "d_acuan": 0.5
 },
 "ktp_fulan.jpg": {
  "terbaik": "B_otsu",
  "d_conf": -0.91,
  "d_acuan": 0.1
 },
 "ktp_fulani.jpg": {
  "terbaik": "B_otsu",
  "d_conf": 7.8,
  "d_acuan": 0.0
 },
 "ktp_jack.jpg": {
  "terbaik": "F_adaptive_morfologi",
  "d_conf": -9.14,
  "d_acuan": 0.0
 }
}


In [20]:
# Pembanding: hasil Modul 7 (tanpa morfologi) pada acuan yang sama
pembanding = {}
for berkas in BERKAS:
    rows = acuan[berkas]['rows']
    pembanding[berkas] = {m: round(sum(1 for r in rows if r['hasil'].get(f'{m}_ok'))/len(rows), 4)
                          for m in ('Berwarna', 'Otsu', 'Adaptive')}
R['ktp_pembanding'] = pembanding
print(json.dumps(pembanding, indent=1))

{
 "ktp_adam.jpg": {
  "Berwarna": 0.7,
  "Otsu": 0.0,
  "Adaptive": 0.0
 },
 "ktp_fulan.jpg": {
  "Berwarna": 1.0,
  "Otsu": 1.0,
  "Adaptive": 0.9
 },
 "ktp_fulani.jpg": {
  "Berwarna": 1.0,
  "Otsu": 0.7,
  "Adaptive": 0.2
 },
 "ktp_jack.jpg": {
  "Berwarna": 0.6,
  "Otsu": 0.6,
  "Adaptive": 0.9
 }
}


In [21]:
# Gambar bukti tugas: tahapan morfologi dan perbandingan sebelum/sesudah
bgr_a = cv2.imread(os.path.join(KTP_DIR, 'ktp_adam.jpg'))
gray_a = cv2.cvtColor(bgr_a, cv2.COLOR_BGR2GRAY)
v = varian(gray_a, bgr_a)
fig, ax = plt.subplots(1, 5, figsize=(15, 3.6))
for a, g, t in zip(ax, [gray_a, v['B_otsu'], v['C_otsu_morfologi'], v['E_tophat_otsu_morfologi'],
                        v['G_kanal_biru_morfologi']],
                   ['A grayscale', 'B Otsu', 'C Otsu + morfologi', 'E top-hat + Otsu + morfologi',
                    'G kanal biru + morfologi']):
    a.imshow(g, cmap='gray'); a.set_title(t, fontsize=9); a.set_xticks([]); a.set_yticks([])
fig.suptitle('Tahapan praolah morfologi pada KTP', fontsize=10)
simpan(fig, '16_ktp_tahapan.png')
plt.show()

fig, ax = plt.subplots(2, 4, figsize=(13, 6.4))
for k, berkas in enumerate(BERKAS):
    b = cv2.imread(os.path.join(KTP_DIR, berkas))
    g = cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)
    vv = varian(g, b)
    ax[0, k].imshow(g, cmap='gray'); ax[0, k].set_title(f'{berkas} - awal', fontsize=8)
    ax[1, k].imshow(vv[hasil_ktp[berkas]['terbaik']], cmap='gray')
    ax[1, k].set_title(hasil_ktp[berkas]['terbaik'], fontsize=8)
    for a in (ax[0, k], ax[1, k]):
        a.set_xticks([]); a.set_yticks([])
fig.suptitle('Perbandingan citra sebelum dan sesudah praolah morfologi', fontsize=10)
simpan(fig, '17_ktp_sebelum_sesudah.png')
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
x = np.arange(len(BERKAS)); w = 0.36
label = [b.replace('ktp_', '').replace('.jpg', '') for b in BERKAS]
conf_awal = [hasil_ktp[b]['varian']['A_grayscale']['conf'] for b in BERKAS]
conf_baik = [hasil_ktp[b]['varian'][hasil_ktp[b]['terbaik']]['conf'] for b in BERKAS]
acu_awal = [hasil_ktp[b]['varian']['A_grayscale']['proporsi_acuan'] for b in BERKAS]
acu_baik = [hasil_ktp[b]['varian'][hasil_ktp[b]['terbaik']]['proporsi_acuan'] for b in BERKAS]
ax[0].bar(x - w/2, conf_awal, w, label='sebelum'); ax[0].bar(x + w/2, conf_baik, w, label='sesudah')
ax[0].set_xticks(x); ax[0].set_xticklabels(label); ax[0].set_ylabel('rerata conf')
ax[0].legend(fontsize=8); ax[0].set_title('Keyakinan OCR', fontsize=9)
ax[1].bar(x - w/2, acu_awal, w, label='sebelum'); ax[1].bar(x + w/2, acu_baik, w, label='sesudah')
ax[1].set_xticks(x); ax[1].set_xticklabels(label); ax[1].set_ylabel('proporsi acuan ditemukan')
ax[1].legend(fontsize=8); ax[1].set_title('Kata acuan terbaca', fontsize=9)
simpan(fig, '18_ktp_grafik.png')
plt.show()

In [22]:
# Sel terakhir: tulis results.json (sumber seluruh angka laporan)
with open(os.path.join(BASE, 'results.json'), 'w', encoding='utf-8') as f:
    json.dump(R, f, ensure_ascii=False, indent=1)
print('results.json ditulis:', os.path.join(BASE, 'results.json'))
print('kunci:', list(R.keys()))

results.json ditulis: C:\Kuliah\Tingkat 3\PCVK\Week 9\results.json
kunci: ['nama', 'nim', 'N', 'lingkungan', 'waktu', 'citra', 'dilasi', 'erosi', 'gabungan', 'lanjut', 'ktp', 'ktp_kanal', 'ktp_pembanding']
